In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer

# 1. Đọc và làm sạch dữ liệu
df = pd.read_csv('C:\Học sâu\Dataset\data\processed\FINAL_MERGED_3FILES_20251224_184903_LABELED_ONLY_20251224_190142_TEENCODE_CONTEXT_150_fixed.csv')
df = df.dropna(subset=['training_text', 'label'])  # Loại bỏ null
df['training_text'] = df['training_text'].astype(str)  # Đảm bảo là string

# 2. Mã hóa label nếu cần (kiểm tra kiểu dữ liệu trước)
if df['label'].dtype == 'object':
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])

# 3. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# 4. Chia dữ liệu
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['training_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# 5. Tokenize (giảm max_length xuống 128)
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=256,  # Thay đổi từ 256
        return_tensors="pt"
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

# 6. Dataset class
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)  # ← Đổi thành torch.long
        return item

    def __len__(self):
        return len(self.labels)

# Tạo lại dataset
train_dataset = ToxicDataset(train_encodings, train_labels)
val_dataset = ToxicDataset(val_encodings, val_labels)

# Kiểm tra
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")
print(f"Sample: {train_dataset[0]}")

c:\Users\tran thien\.conda\envs\linear_regression\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train size: 3113, Val size: 779
Sample: {'input_ids': tensor([    0,   702,     8,   572,   237,    14,  2902, 15022,  1973,    43,
          385,    91,   413,   138,   430,   295,   110,   235,  3253,  1362,
        63117,  3787,  2090, 12675,   884,  1494,  1529,  1388,  1204,     2,
         2888,   572,  5262,   985,   351,    94,  1388, 16174,  1187,  7390,
            5,     2,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
          

In [2]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Tính trọng số dựa trên tập nhãn huấn luyện
# 'balanced' sẽ tự động gán trọng số cao cho nhãn ít và thấp cho nhãn nhiều
weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to('cuda')

print(f"Trọng số các lớp [0, 1, 2] là: {weights}")
# Giải thích: Trọng số càng cao, model càng bị "phạt" nặng nếu đoán sai lớp đó.

Trọng số các lớp [0, 1, 2] là: [0.71464646 1.22946288 1.27009384]


In [3]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch
import numpy as np

# ============================================================================
# 1. KHỞI TẠO MÔ HÌNH
# ============================================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=3  # 3 nhãn: [0, 1, 2]
)
model.config.problem_type = "single_label_classification"

# ============================================================================
# 2. CLASS WEIGHTS (từ kết quả tính toán của bạn)
# ============================================================================
# Trọng số các lớp [0, 1, 2]: [0.70813397, 1.20465116, 1.31974522]
class_weights = torch.tensor(
    [0.70813397, 1.20465116, 1.21974522],
    dtype=torch.float32
).to('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Class weights đã load: {class_weights}")
print(f"Device: {class_weights.device}")

# ============================================================================
# 3. HÀM TÍNH METRICS
# ============================================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Tính F1-macro và Accuracy
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

# ============================================================================
# 4. CUSTOM TRAINER VỚI WEIGHTED LOSS
# ============================================================================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Override loss function để áp dụng class weights
        """
        # Lấy labels từ inputs
        labels = inputs.pop("labels")

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.logits  # Sử dụng .logits thay vì .get("logits")

        # Chuyển labels về kiểu Long trước khi tính loss
        labels = labels.to(torch.long)

        # Tính loss với class weights
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# ============================================================================
# 5. TRAINING ARGUMENTS (Cấu hình huấn luyện tối ưu)
# ============================================================================
training_args = TrainingArguments(
    # Đường dẫn lưu kết quả
    output_dir='./phobert_toxic_results',

    # Epochs và batch size
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,  # Eval batch có thể lớn hơn

    # Gradient accumulation (nếu GPU bị out of memory, tăng lên 2 hoặc 4)
    gradient_accumulation_steps=1,

    # Learning rate và optimizer
    learning_rate=1e-5,
    weight_decay=0.1,
    warmup_ratio=0.2,  # Warmup 10% số bước đầu

    # Evaluation và saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,  # Chỉ giữ 2 checkpoint tốt nhất
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',  # Chọn model tốt nhất theo F1-macro
    greater_is_better=True,
    # logging
    logging_dir='./logs',
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",  # Tắt wandb/tensorboard nếu không dùng

    # Performance
    fp16=torch.cuda.is_available(),  # Sử dụng mixed precision nếu có GPU
    dataloader_num_workers=0,

    # Reproducibility
    seed=42,
)

# ============================================================================
# 6. KHỞI TẠO TRAINER
# ============================================================================
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# ============================================================================
# 7. HUẤN LUYỆN MÔ HÌNH
# ============================================================================
print("=" * 80)
print("BẮT ĐẦU HUẤN LUYỆN")
print("=" * 80)

trainer.train()

# ============================================================================
# 8. ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP VALIDATION
# ============================================================================
print("\n" + "=" * 80)
print("ĐÁNH GIÁ TRÊN TẬP VALIDATION")
print("=" * 80)

eval_results = trainer.evaluate()
print(f"\nKết quả đánh giá:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

# ============================================================================
# 9. DỰ ĐOÁN VÀ PHÂN TÍCH CHI TIẾT
# ============================================================================
predictions = trainer.predict(val_dataset)
y_pred = predictions.predictions.argmax(-1)
y_true = predictions.label_ids

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)
print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1', 'Class 2']))

# ============================================================================
# 10. LƯU MÔ HÌNH
# ============================================================================
trainer.save_model("./phobert_toxic_final_model")
print("\n✅ Đã lưu model tại: ./phobert_toxic_final_model")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights đã load: tensor([0.7081, 1.2047, 1.2197], device='cuda:0')
Device: cuda:0
BẮT ĐẦU HUẤN LUYỆN


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.985000,0.910260,0.539153,0.533068,0.531897


KeyboardInterrupt: 